<a href="https://colab.research.google.com/github/Protein-Function-Prediction/COMP3608ProteinFunctionPrediction/blob/Classical-Model/classical_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COMP 3608 – Protein Function Classification Pipeline

## Problem Classification & Algorithm Justification

### Problem Classification
This pipeline addresses a **multi-class classification problem**. The goal is to predict one of several discrete functional classes or GO terms for a given protein based on its biophysical descriptors.

### Algorithm Selection and Justification
For this multi-class protein function classification task, two powerful and relevant machine learning algorithms have been selected:

1.  **Logistic Regression (LR)**:
    *   **Justification**: LR is chosen as a robust and interpretable baseline. It is a linear model that performs well when class boundaries are approximately linear. Its probabilistic nature (providing class probabilities) and computational efficiency make it a suitable first approach. It's considered a fundamental state-of-the-art method for its simplicity and effectiveness in many contexts, especially for establishing a performance floor.

2.  **Support Vector Machine (SVM) with RBF Kernel**:
    *   **Justification**: SVM with a Radial Basis Function (RBF) kernel is selected for its ability to model non-linear decision boundaries. Protein biophysical feature spaces are often complex and non-linearly separable, making SVM-RBF particularly relevant. SVMs are well-regarded for their strong theoretical foundation (maximal margin classification) and have demonstrated state-of-the-art performance in various bioinformatics and classification tasks, especially in high-dimensional feature spaces where they can avoid local minima issues that plague some other non-linear models. The RBF kernel allows the model to capture intricate relationships within the data without explicitly mapping features to a higher-dimensional space.

Both algorithms are widely recognized, well-understood, and highly relevant to complex classification problems.

This section classifies the problem as a multi-class classification and justifies the selection of Logistic Regression and Support Vector Machine (SVM) with an RBF kernel as the chosen algorithms, highlighting their strengths for this type of task.

## Objective Function (Z)

The primary objective for optimizing and evaluating our models is the **Macro-F1 Score**. This metric is particularly crucial for multi-class classification problems, especially when dealing with imbalanced datasets, as it provides an unweighted mean of the F1 score for each class, thereby giving equal importance to all classes (minority and majority alike). This prevents models from trivially performing well by only classifying the majority class.

Mathematically, the Macro-F1 score (Z) is defined as follows:

Given $K$ classes, the F1 score for each class $k$ is calculated as:

$$F1_k = 2 \times \frac{\text{Precision}_k \times \text{Recall}_k}{\text{Precision}_k + \text{Recall}_k}$$

Where:
*   **Precision$_k$** = $\frac{\text{True Positives}_k}{\text{True Positives}_k + \text{False Positives}_k}$
*   **Recall$_k$** = $\frac{\text{True Positives}_k}{\text{True Positives}_k + \text{False Negatives}_k}$

The **Macro-F1 score (Z)** is then the arithmetic mean of the F1 scores for all $K$ classes:

$$Z = \text{Macro-F1} = \frac{1}{K} \sum_{k=1}^{K} F1_k$$

This objective function accurately captures the problem's goal of achieving robust and balanced classification performance across all protein functional classes, rather than being biased towards larger classes.

## Experimental Design

Our experimental design is structured to ensure a robust and fair evaluation of the protein function classification models, adhering to machine learning best practices and directly addressing the problem's complexities.

### Datasets Selection
We have selected **three distinct, relevant, and real-world protein datasets** (df1, df2, df3) to ensure comprehensive testing across different types of protein function classification tasks:
*   **df1:** Focuses on broad structural/functional types, representing a fundamental classification task.
*   **df2:** Utilizes GO cellular component terms, providing a multi-label classification challenge with a higher number of classes.
*   **df3:** Uses a subset of GO molecular function terms, offering another granular classification task.

### Soundness and Best Practices
1.  **Train-Test Split:** Each dataset is first split into 80% training and 20% testing sets using `train_test_split`. This ensures that models are evaluated on unseen data, providing an unbiased estimate of generalization performance.
2.  **Stratified Splitting:** Crucially, `stratify=y` is used during the train-test split for all datasets. This maintains the same proportion of target classes in both the training and testing sets as in the original dataset, which is vital for multi-class classification and prevents misleading results due to skewed class distributions in splits.
3.  **Cross-Validation:** For a more robust estimate of model performance and to assess generalization stability, **5-Fold Stratified Cross-Validation** is employed. This technique repeatedly splits the training data, training on a subset and validating on another, mitigating the impact of any single train-test split and providing mean and standard deviation of performance metrics.
4.  **Hyperparameter Tuning:** `GridSearchCV` is utilized with a 3-fold stratified cross-validation on the training data to systematically search for the optimal hyperparameters for both Logistic Regression and SVM. This ensures that the models are evaluated at their best possible configuration, preventing suboptimal performance due to un-tuned parameters.
5.  **Addressing Class Imbalance:** Given the inherent class imbalances often found in biological datasets, two strategies are employed:
    *   `class_weight='balanced'` is incorporated into both Logistic Regression and SVM models. This automatically adjusts weights inversely proportional to class frequencies, giving more importance to minority classes.
    *   **SMOTE (Synthetic Minority Over-sampling Technique)** is applied specifically to the training data of `df2` (the most imbalanced dataset). SMOTE generates synthetic samples for minority classes, thereby balancing the class distribution and helping models learn patterns from under-represented classes.
6.  **Performance Metrics:** The **Macro-F1 score** is selected as the primary optimization and evaluation metric. As previously defined, Macro-F1 provides an unweighted average of F1 scores per class, making it robust against class imbalance and giving equal importance to the accurate classification of all protein functions. Accuracy is also reported as a supplementary metric.

This design ensures that our evaluation is comprehensive, fair, and directly addresses the challenges of multi-class protein function classification in real-world scenarios.

In [30]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import os
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
TEST_SIZE = 0.20
OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


This cell imports necessary libraries for data manipulation, machine learning, plotting, and file system operations. It also sets global constants like `RANDOM_STATE`, `TEST_SIZE`, and `OUTPUT_DIR`.

In [28]:
import sys
!{sys.executable} -m pip install imblearn

This cell ensures that the `imblearn` library, which is used for handling imbalanced datasets (e.g., SMOTE), is installed in the Colab environment.

In [11]:

# DATA LOADING & PREPROCESSING

def parse_dotted_float(s, lo, hi):
    """
    df1 stores floats with locale-style dot separators (e.g. '20.362.946...' → 20.36).
    Strategy: strip all dots, then try every decimal position until value falls in [lo, hi].
    """
    try:
        digits = str(s).replace(".", "").lstrip("0") or "0"
        n = len(digits)
        for i in range(1, n + 1):
            v = (float(digits[:i] + "." + digits[i:])
                 if i < n else float(digits[:i]))
            if lo <= v <= hi:
                return v
        return np.nan
    except Exception:
        return np.nan


def load_df1(path):
    """
    df1: 5-class protein functional annotation.
    Features: Net_Charge, Sequence_Length + parsed biophysical string columns.
    Target: Class_enc (0-4)
    """
    df = pd.read_csv(path)

    # Parse locale-encoded string columns
    df["MW_kDa"] = df["Molecular_Weight"].apply(lambda x: parse_dotted_float(x, 1,   350))
    df["pI"] = df["Isoelectric_Point"].apply(lambda x: parse_dotted_float(x, 1,   14))
    df["GRAVY"] = df["Hydrophobicity"].apply(lambda x: parse_dotted_float(x, -5,   5))
    df["Polar"] = df["Polar_Ratio"].apply(lambda x: parse_dotted_float(x,   0,  100))
    df["NonPolar"] = df["NonPolar_Ratio"].apply(lambda x: parse_dotted_float(x,  0,  100))

    feature_cols = ["Net_Charge", "Sequence_Length", "MW_kDa", "pI", "Polar", "NonPolar"]
    df = df[feature_cols + ["Class_enc", "Class"]].dropna()

    X = df[feature_cols].values
    y = df["Class_enc"].values
    labels = df["Class"].unique().tolist()
    label_names = [df[df["Class_enc"] == i]["Class"].iloc[0]
                   for i in sorted(df["Class_enc"].unique())]
    return X, y, label_names


def load_df2(path):
    """
    df2: 20-class GO cellular component labels.
    Features: 20 numeric biophysical descriptors.
    Target: GO_label_enc (0-19)
    """
    df = pd.read_csv(path, on_bad_lines='skip')
    bio_cols = [
        "seq_length", "mol_weight", "pI", "gravy", "instability",
        "aromaticity", "helix", "turn", "sheet",
        "aa_A","aa_C","aa_D","aa_E","aa_F","aa_G","aa_H","aa_I",
        "aa_K","aa_L","aa_M","aa_N","aa_P","aa_Q","aa_R","aa_S",
        "aa_T","aa_V","aa_W","aa_Y"
    ]
    df = df[bio_cols + ["GO_label_enc", "GO_label"]].dropna()
    X = df[bio_cols].values
    y = df["GO_label_enc"].values
    label_names = [df[df["GO_label_enc"] == i]["GO_label"].iloc[0]
                   for i in sorted(df["GO_label_enc"].unique())]
    return X, y, label_names


def load_df3(path, top_n=10):
    """
    df3: Reduce to top-N GO molecular function classes for tractable SVM training.
    Features: same biophysical descriptors as df2.
    Target: re-encoded label (0 to top_n-1)
    """
    df = pd.read_csv(path)
    bio_cols = [
        "seq_length", "mol_weight", "pI", "gravy", "instability",
        "aromaticity", "helix", "turn", "sheet",
        "aa_A","aa_C","aa_D","aa_E","aa_F","aa_G","aa_H","aa_I",
        "aa_K","aa_L","aa_M","aa_N","aa_P","aa_Q","aa_R","aa_S",
        "aa_T","aa_V","aa_W","aa_Y"
    ]
    top_go = df["GO_id"].value_counts().nlargest(top_n).index
    df = df[df["GO_id"].isin(top_go)][bio_cols + ["GO_id"]].dropna()

    le = LabelEncoder()
    y = le.fit_transform(df["GO_id"].values)
    label_names = list(le.classes_)
    X = df[bio_cols].values
    return X, y, label_names

This cell defines helper functions (`parse_dotted_float`, `load_df1`, `load_df2`, `load_df3`) responsible for loading and preprocessing the three different protein datasets. Each function handles specific data parsing and feature selection relevant to its dataset.

In [14]:
def build_lr(multi="multinomial", C=1.0, max_iter=1000):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            multi_class=multi, C=C,
            max_iter=max_iter, random_state=RANDOM_STATE, n_jobs=-1,
            class_weight='balanced'
        ))
    ])


def build_svm(C=1.0, gamma="scale", kernel="rbf"):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            C=C, gamma=gamma, kernel=kernel,
            decision_function_shape="ovr",
            random_state=RANDOM_STATE,
            class_weight='balanced'
        ))
    ])

This cell defines functions (`build_lr`, `build_svm`) to construct the machine learning pipelines for Logistic Regression and Support Vector Machines. These pipelines include `StandardScaler` for feature scaling and configure the respective models with parameters like `multi_class`, `C`, `max_iter`, `gamma`, and `class_weight='balanced'`.

In [4]:

# EVALUATION HELPER


def evaluate(pipeline, X_train, X_test, y_train, y_test,
             label_names, dataset_name, model_name):
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    report = classification_report(y_test, y_pred, target_names=label_names, zero_division=0)

    print(f"\n{'='*60}")
    print(f"  Dataset : {dataset_name}")
    print(f"  Model   : {model_name}")
    print(f"{'='*60}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro-F1 : {f1:.4f}")
    print(f"\n{report}")

    return {"dataset": dataset_name, "model": model_name,
            "accuracy": acc, "macro_f1": f1,
            "y_test": y_test, "y_pred": y_pred,
            "label_names": label_names}


This cell defines the `evaluate` helper function. This function takes a trained pipeline and test data, makes predictions, and calculates key performance metrics such as accuracy and Macro-F1 score, also printing a detailed classification report.

In [5]:
# CROSS-VALIDATION HELPER

def cv_score(pipeline, X, y, cv=5, scoring="f1_macro"):
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipeline, X, y, cv=skf, scoring=scoring, n_jobs=-1)
    return scores.mean(), scores.std()

This cell defines the `cv_score` helper function, which performs stratified k-fold cross-validation on a given pipeline and dataset. It returns the mean and standard deviation of the specified scoring metric (e.g., Macro-F1) across the folds.

In [6]:
# PLOTTING

def plot_confusion_matrices(results, filename):
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, r in zip(axes, results):
        cm = confusion_matrix(r["y_test"], r["y_pred"])
        # normalise for readability
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        sns.heatmap(cm_norm, annot=(len(r["label_names"]) <= 10),
                    fmt=".2f", cmap="Blues",
                    xticklabels=r["label_names"],
                    yticklabels=r["label_names"],
                    ax=ax, linewidths=0.5, cbar=False)
        ax.set_title(f"{r['dataset']}\n{r['model']}", fontsize=12, fontweight="bold")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.tick_params(axis="y", rotation=0, labelsize=7)

    plt.suptitle("Normalised Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(filename, bbox_inches="tight", dpi=120)
    plt.close()
    print(f"  Saved → {filename}")


def plot_summary_bar(summary_df, filename):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    palette = {"Logistic Regression": "#4C72B0", "SVM (RBF)": "#DD8452"}

    for ax, metric in zip(axes, ["accuracy", "macro_f1"]):
        data = summary_df.pivot(index="dataset", columns="model", values=metric)
        data.plot(kind="bar", ax=ax, color=[palette[c] for c in data.columns],
                  edgecolor="black", linewidth=0.6, width=0.55)
        ax.set_title(metric.replace("_", " ").title(), fontsize=13, fontweight="bold")
        ax.set_ylabel("Score")
        ax.set_xlabel("")
        ax.set_ylim(0, 1.05)
        ax.tick_params(axis="x", rotation=20)
        ax.legend(title="Model", fontsize=9)
        for bar in ax.patches:
            h = bar.get_height()
            if h > 0:
                ax.annotate(f"{h:.3f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                            xytext=(0, 3), textcoords="offset points",
                            ha="center", va="bottom", fontsize=8)

    plt.suptitle("LR vs SVM – Performance Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(filename, bbox_inches="tight", dpi=120)
    plt.close()
    print(f"  Saved → {filename}")


def plot_cv_comparison(cv_results, filename):
    """Bar chart of 5-fold CV Macro-F1 with std error bars."""
    labels  = [f"{r['dataset']}\n{r['model']}" for r in cv_results]
    means = [r["mean"] for r in cv_results]
    stds = [r["std"]  for r in cv_results]
    colors  = ["#4C72B0" if "LR" in r["model"] else "#DD8452" for r in cv_results]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(labels, means, yerr=stds, capsize=5,
                  color=colors, edgecolor="black", linewidth=0.6, width=0.55)
    ax.set_ylabel("Macro-F1 (5-fold CV)")
    ax.set_ylim(0, 1.1)
    ax.set_title("5-Fold Cross-Validation – Macro-F1 with Std Dev",
                 fontsize=13, fontweight="bold")
    ax.tick_params(axis="x", labelsize=8)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2,
                m + s + 0.02, f"{m:.3f}±{s:.3f}",
                ha="center", va="bottom", fontsize=8)

    # Custom legend
    from matplotlib.patches import Patch
    legend_els = [Patch(color="#4C72B0", label="Logistic Regression"),
                  Patch(color="#DD8452", label="SVM (RBF)")]
    ax.legend(handles=legend_els, fontsize=9)
    plt.tight_layout()
    plt.savefig(filename, bbox_inches="tight", dpi=120)
    plt.close()
    print(f"  Saved → {filename}")

This cell contains functions for generating various plots to visualize the experimental results. `plot_confusion_matrices` creates confusion matrices for each model and dataset, `plot_summary_bar` compares model performance (accuracy and Macro-F1) across datasets, and `plot_cv_comparison` visualizes cross-validation Macro-F1 scores.

In [32]:
# MAIN EXPERIMENT

# Hyperparameter grids for GridSearchCV
lr_param_grid = {
    "lr__C": [0.01, 0.1, 1, 10, 100],
    "lr__solver": ["liblinear", "saga"]
}

svm_param_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", "auto", 0.1, 1]
}

def main():
    print("\n" + "="*60)
    print("  COMP 3608 – SVM & LR Protein Classification Pipeline")
    print("="*60)

    #  Load datasets
    print("\n[1/4] Loading datasets …")
    X1, y1, labels1 = load_df1("/content/df1_cleaned.csv")
    X2, y2, labels2 = load_df2("/content/df2_cleaned.csv")
    X3, y3, labels3 = load_df3("/content/df3_cleaned.csv", top_n=10)

    print(f"  df1 → X:{X1.shape}  classes:{len(labels1)}")
    print(f"  df2 → X:{X2.shape}  classes:{len(labels2)}")
    print(f"  df3 → X:{X3.shape}  classes:{len(labels3)}")

    # Train / Test split
    print("\n[2/4] Splitting 80/20 …")
    splits = {}
    for name, X, y in [("df1", X1, y1), ("df2", X2, y2), ("df3", X3, y3)]:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

        # Apply SMOTE to df2 training data only
        if name == "df2":
            print(f"  Applying SMOTE to {name} training set (original shape: {X_tr.shape})")
            smote = SMOTE(random_state=RANDOM_STATE)
            X_tr, y_tr = smote.fit_resample(X_tr, y_tr)
            print(f"  {name} training set shape after SMOTE: {X_tr.shape}")

        splits[name] = (X_tr, X_te, y_tr, y_te)
        print(f"  {name}: train={X_tr.shape[0]}  test={X_te.shape[0]}")

    # Model definitions
    datasets_info = [
        ("df1 – 5-Class Protein Type", "df1", labels1),
        ("df2 – 20-Class GO Label", "df2", labels2),
        ("df3 – 10-Class GO Function", "df3", labels3),
    ]

    # Note: SVM on df2 (65k rows) uses linear kernel + balanced for speed;
    # RBF on df3 is tractable at ~10k rows.
    model_factories = {
        "df1": [
            ("Logistic Regression", build_lr()),
            ("SVM (RBF)", build_svm(C=1.0)),
        ],
        "df2": [
            ("Logistic Regression", build_lr(max_iter=2000)),
            ("SVM (RBF)", build_svm(C=1.0)),
        ],
        "df3": [
            ("Logistic Regression", build_lr()),
            ("SVM (RBF)", build_svm(C=1.0)),
        ],
    }

    #  Run experiments
    print("\n[3/4] Training & evaluating …")
    all_results  = []
    summary_rows = []
    cv_results   = []

    for ds_label, ds_key, ds_labels in datasets_info:
        X_tr, X_te, y_tr, y_te = splits[ds_key]

        # df2 is large – subsample train for SVM to keep runtime feasible
        if ds_key == "df2" and X_tr.shape[0] > 15000:
            rng = np.random.RandomState(RANDOM_STATE)
            idx = rng.choice(X_tr.shape[0], 15000, replace=False)
            X_tr_svm, y_tr_svm = X_tr[idx], y_tr[idx]
        else:
            X_tr_svm, y_tr_svm = X_tr, y_tr

        for model_name, pipe in model_factories[ds_key]:
            X_train_use = X_tr_svm if ("SVM" in model_name and ds_key == "df2") else X_tr
            y_train_use = y_tr_svm if ("SVM" in model_name and ds_key == "df2") else y_tr

            # Hyperparameter tuning with GridSearchCV
            print(f"  Tuning {model_name} for {ds_label}…")
            param_grid = lr_param_grid if "Logistic Regression" in model_name else svm_param_grid
            grid_search = GridSearchCV(pipe, param_grid, cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE), scoring="f1_macro", n_jobs=-1, verbose=0)
            grid_search.fit(X_train_use, y_train_use)
            best_pipeline = grid_search.best_estimator_

            print(f"    Best parameters: {grid_search.best_params_}")

            r = evaluate(best_pipeline, X_train_use, X_te,
                         y_train_use, y_te,
                         ds_labels, ds_label, model_name)
            all_results.append(r)
            summary_rows.append({
                "dataset":  ds_label,
                "model":    model_name,
                "accuracy": r["accuracy"],
                "macro_f1": r["macro_f1"],
            })

            # 5-fold CV on full training set (using the best pipeline from GridSearchCV)
            print(f"  Running 5-fold CV: {ds_label} | {model_name} …")
            # Use a 5k subsample for CV on large datasets
            if X_tr.shape[0] > 10000:
                rng = np.random.RandomState(RANDOM_STATE)
                idx = rng.choice(X_tr.shape[0], 10000, replace=False)
                Xcv, ycv = X_tr[idx], y_tr[idx]
            else:
                Xcv, ycv = X_tr, y_tr

            cv_mean, cv_std = cv_score(best_pipeline, Xcv, ycv, cv=5, scoring="f1_macro")
            cv_results.append({
                "dataset": ds_label, "model": model_name,
                "mean": cv_mean, "std": cv_std,
            })
            print(f"    CV Macro-F1: {cv_mean:.4f} ± {cv_std:.4f}")

    # Summary table
    summary_df = pd.DataFrame(summary_rows)
    print("\n" + "="*60)
    print("  SUMMARY TABLE")
    print("="*60)
    print(summary_df.to_string(index=False, float_format="{:.4f}"))

    # Plots
    print("\n[4/4] Generating plots …")

    # Confusion matrices
    plot_confusion_matrices(
        all_results,
        f"{OUTPUT_DIR}/confusion_matrices.png"
    )

    # Bar chart comparison
    plot_summary_bar(
        summary_df,
        f"{OUTPUT_DIR}/performance_comparison.png"
    )

    # CV comparison
    plot_cv_comparison(
        cv_results,
        f"{OUTPUT_DIR}/cv_comparison.png"
    )

    # Save summary CSV
    summary_df.to_csv(f"{OUTPUT_DIR}/results_summary.csv", index=False)
    print(f" Saved → {OUTPUT_DIR}/results_summary.csv")

    print("\n✓ Pipeline complete.\n")


This cell contains the `main` function, which orchestrates the entire experimental pipeline. It loads data, performs train-test splits, applies SMOTE where necessary, defines model factories, performs hyperparameter tuning using `GridSearchCV`, evaluates models, runs cross-validation, generates a summary table, and saves all results and plots.

In [26]:
if __name__ == "__main__" or True:   # True → runs in notebook context too
    main()


  COMP 3608 – SVM & LR Protein Classification Pipeline

[1/4] Loading datasets …
  df1 → X:(16000, 6)  classes:5
  df2 → X:(65205, 29)  classes:20
  df3 → X:(10408, 29)  classes:10

[2/4] Splitting 80/20 …
  df1: train=12800  test=3200
  Applying SMOTE to df2 training set (original shape: (52164, 29))
  df2 training set shape after SMOTE: (336080, 29)
  df2: train=336080  test=13041
  df3: train=8326  test=2082

[3/4] Training & evaluating …

  Dataset : df1 – 5-Class Protein Type
  Model   : Logistic Regression
  Accuracy : 0.2022
  Macro-F1 : 0.1902

              precision    recall  f1-score   support

      Enzyme       0.20      0.25      0.22       647
      Others       0.20      0.31      0.24       637
    Receptor       0.20      0.08      0.11       625
  Structural       0.20      0.26      0.22       646
 Transporter       0.23      0.11      0.15       645

    accuracy                           0.20      3200
   macro avg       0.20      0.20      0.19      3200
weight

This cell is the entry point for executing the `main` function. The `if __name__ == '__main__'` block ensures that `main()` is called when the script is run directly. The `or True` part allows it to execute within the Colab notebook context as well.

## Insights and Conclusions

This project embarked on classifying protein functions using biophysical descriptors, employing Logistic Regression (LR) and Support Vector Machines (SVM). Our rigorous experimental design, including stratified splitting, cross-validation, hyperparameter tuning, and strategies for class imbalance (class weights, SMOTE), allowed us to draw several key insights.

### Detailed Experimental Results

**Dataset 1 (df1 – 5-Class Protein Type):**

*   **Logistic Regression (LR):** Achieved an average Macro-F1 of approximately **0.190** on the test set, with a 5-fold CV Macro-F1 of **0.189 ± 0.010**. The individual class F1-scores were uniformly low, indicating that the linear model struggled to find clear separation boundaries for these protein types. The balanced class weights helped prevent a bias towards any single class but couldn't significantly boost overall performance. This suggests that the relationship between biophysical features and these broad functional types might be inherently complex and non-linear, or that the features themselves are not highly discriminative for this classification task.

*   **SVM (RBF):** Showed slightly better, but still low, performance with a Macro-F1 of **0.194** on the test set and a 5-fold CV Macro-F1 of **0.192 ± 0.007**. While SVM with an RBF kernel is designed to capture non-linear relationships, the marginal improvement over LR indicates that even a non-linear boundary wasn't sufficient to classify these 5 protein types effectively with the given features. Both models performed barely better than random chance (which would be 0.20 for 5 classes if evenly distributed).

**Dataset 2 (df2 – 20-Class GO Label):**

*   **Logistic Regression (LR):** Recorded an extremely low Macro-F1 of **0.011** on the test set, with a 5-fold CV Macro-F1 of **0.010 ± 0.000**. This result is barely above zero and signifies almost complete failure in classifying the 20 GO cellular component terms. Individual class performance was negligible for most classes.

*   **SVM (RBF):** Similarly, SVM performed very poorly, yielding a Macro-F1 of **0.020** on the test set and a 5-fold CV Macro-F1 of **0.021 ± 0.003**. Despite the use of `class_weight='balanced'` and **SMOTE** (which massively oversampled the training data to balance classes), the performance remained abysmal. The subsampling of the training data for SVM (to keep training tractable) might have played a minor role, but the fundamental issue appears deeper. This result strongly indicates that the biophysical features provided are largely non-discriminative for these 20 GO cellular component terms, or that the problem is significantly more complex than what LR and non-linear SVM can handle.

**Dataset 3 (df3 – 10-Class GO Function):**

*   **Logistic Regression (LR):** Achieved a Macro-F1 of **0.425** on the test set, with a 5-fold CV Macro-F1 of **0.424 ± 0.007**. This is a moderate performance, indicating that LR can find some linear patterns for these 10 GO molecular function terms.

*   **SVM (RBF):** Demonstrated the best performance across all experiments, with a Macro-F1 of **0.556** on the test set and a 5-fold CV Macro-F1 of **0.549 ± 0.011**. This substantial improvement over LR (over 13 percentage points in Macro-F1) highlights the RBF kernel's ability to capture non-linear decision boundaries that are present in this dataset. The better performance on `df3` compared to `df1` and `df2` suggests that these 10 GO molecular function terms might be more distinctly characterized by the biophysical features provided.

### Identification of Possible Errors and Their Solutions (During Development)

Throughout the development of this pipeline, several critical issues were identified and resolved, which significantly enhanced the robustness and reliability of our results:

2.  **Class Imbalance Mitigation:** Recognizing the inherent imbalance in biological datasets, `class_weight='balanced'` was integrated into both LR and SVM models. For the severely imbalanced `df2`, **SMOTE** was additionally applied to the training data. While SMOTE significantly balanced the training sets, its impact on `df2`'s final performance was limited, suggesting fundamental data limitations rather than just class distribution issues.

5.  **Robust Evaluation:** The adoption of **Macro-F1** as the primary metric, combined with 5-fold stratified cross-validation, provided a much more reliable and balanced assessment of model performance, especially crucial for multi-class, imbalanced scenarios. The initial reliance solely on accuracy or a single train-test split could have led to misleading conclusions.

### Usefulness to Stakeholders

For a non-technical stakeholder, the key takeaways from this experiment are:

*   **Difficulty of Protein Function Prediction:** Predicting a protein's function solely from its basic biophysical properties is a challenging task. Our models, while employing standard best practices, showed varying degrees of success depending on the dataset.

*   **Dataset-Specific Performance:** The ability to classify protein functions varies significantly across different types of functional classifications. For instance:
    *   **Broad Protein Types (df1):** Both simple (LR) and more complex (SVM) models struggled, performing only slightly better than random guessing. This implies that the features we used are not sufficiently unique to distinguish these 5 broad types.
    *   **Cellular Components (df2):** The models effectively failed here, producing near-zero performance. This is a critical insight, indicating that a different approach is absolutely required for these 20 GO cellular component terms. The biophysical features alone are simply not enough, even with advanced balancing techniques.
    *   **Molecular Functions (df3):** This dataset yielded the most promising results. The SVM model achieved a moderate level of accuracy and F1-score (around 55%), significantly outperforming the simpler Logistic Regression. This suggests that for these specific 10 GO molecular function terms, the biophysical features *do* contain discriminative information, and a non-linear model like SVM can leverage it effectively.

*   **Need for Further Investment:** Given the low performance on `df1` and especially `df2`, relying on these models for critical decisions in those areas would be premature and risky. Further investment is needed to:
    *   **Explore richer features:** Could we incorporate sequence motifs, evolutionary conservation, or structural motifs? More advanced feature engineering might unlock better performance.
    *   **Employ more advanced models:** Neural Networks (e.g., Deep Learning) or ensemble methods (e.g., Gradient Boosting, Random Forests) are often more powerful at capturing intricate patterns in complex biological data.
    *   **Re-evaluate problem definition for df2:** The 20-class GO cellular component prediction might be too ambitious with the current feature set. Perhaps a hierarchical classification or predicting fewer, broader categories is more feasible.

In summary, while we've built a robust and scientifically sound pipeline, the results highlight the complexity of protein function prediction. We've identified areas where our current approach is effective (df3) and, crucially, areas where significant further research and development are needed (df1 and df2) to achieve reliable predictive power. The methodology ensures that our conclusions are based on solid experimental evidence, guiding future decisions on where to allocate resources for better functional prediction.